<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Adam Glos and Özlem Salehi Köken</a>
        </td>        
</tr></table>

# Max-Cut Problem

Let us consider the following graph.

<img src="../images/graphnocolor.png" width="25%" align="center">

The graph has five edges in total. We will assign a color to each vertex: either blue or red, and count the number of edges that connect vertices with different colors. In the following example, there are two edges connecting vertices that are colored using different colors (blue and red).

<img src="../images/graphcolor2.png" width="25%" align="center">

On the other hand, in the following example, there are four edges connecting the vertices colored using different colors, which is indeed the maximum for a graph with four vertices.

<img src="../images/graphcolor1.png" width="25%" align="center">

## Definition ##

The problem of partitioning the vertices of the graph into two disjoint sets (which is called a cut) such that the number of edges between the two sets (whose endpoints lie in different sets) is maximal is known as the **Max-Cut problem**.

We will focus on the decision version of this problem which is defined as follows:

For a given graph $G$, the problem is to determine whether there exists a cut of size at least $k$. 

It turns out that this problem is challenging (in fact it is **NP**-complete). There are $2^n$ possible colorings, and so a trivial (brute force) search checks each of these cases in the worst case. Here we show that we can solve this problem faster by using Grover's Search algorithm, approximately by making $ \frac{\pi}{4}\sqrt{2^n} \approx 0.8 \times 1.414^n$ oracle queries.

## A simple case: Bipartite graphs

A graph is bipartite if the set of vertices can be divided into two disjoint sets such that each edge connects a vertex from the first set with a vertex from the second set. An example is presented below:

<img src="../images/bipartite.png" width="25%" align="center">

We see that vertex 0 and 1 (or 2 and 3) could form such a set. An example of a graph which is not bipartite is given below.

<img src="../images/nonbipartite.png" width="25%" align="center">

We first focus on bipartite graphs before moving on to the general case.

Bipartite graphs have unique properties such that they admit a 2-coloring, a coloring using two colors in which the endpoints of all the edges are colored with different colors. (Note that now we are talking about a coloring which should make sure that endpoints are colored using different colors. This is different than randomly assigning colors to vertices.) Hence, for bipartite graphs solution to the Max-Cut problem is simply the total number of edges. Therefore, if we can decide efficiently if a graph is bipartite, then we can also efficiently solve the Max-Cut problem for that particular graph.

## Checking bipartiteness

We will check whether a graph is bipartite or not by checking if the endpoints of all the edges are colored with different colors.

We will use the following idea to represent colors of vertices in a quantum circuit. For a graph with $n$ vertices, we will use $n$ qubits. The $i$'th qubit will encode the color of $i$'th vertex as follows: state $\ket{0}$ means the vertex has red color, and state $\ket{1}$ means it has blue color.

### Task 1
    
Let's implement the idea given above. We have a graph with 4 vertices, and so we have a circuit with 4 qubits. 

Represent the following coloring of the given graph in the quantum circuit.

<img src="../images/graphcolor1.png" width="25%" align="center">

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

#Your code here

print(counts)

for qubit in range(4):
    if list(counts.keys())[0][qubit] == '0':
        print("Node",qubit, "is red")
    else:
        print("Node",qubit, "is blue")

[click for our solution](02_max_cut_problem_solutions.ipynb#task1)

## Edge checking 

Next, we will implement a protocol which checks whether an edge has endpoints with different colors. For each edge, we will use a separate qubit, on which we will store a Boolean value: $\ket{1}$ if edge connects vertices with different colors, and $\ket{0}$, otherwise.

We use the XOR function $\oplus$:
$$
x \oplus y = \begin{cases}
1, & x \neq y \\
0, & x = y \\
\end{cases}
$$
where $x$ and $y$ are the colors of vertices and the result is the Boolean value of the edge connecting them.

We can easily implement $x \oplus y=z$ by using two $CNOT$ gates as given below.


In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3)

qc.cx(0,2)
qc.cx(1,2)

print(qc)

               
q_0: ──■───────
       │       
q_1: ──┼────■──
     ┌─┴─┐┌─┴─┐
q_2: ┤ X ├┤ X ├
     └───┘└───┘


Here the first two qubits are $x$ and $y$, and $z$ is the third qubit. 

Let’s verify the correctness of the above circuit by checking all possible coloring (inputs).


In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

colorings = ["00", "01", "10", "11"]

for coloring in colorings:
    print()
    print("Vertex 0 is set to ", coloring[0])
    print("Vertex 1 is set to ", coloring[1])
    
    qc = QuantumCircuit(3, 1)
    
    if coloring[0] == '1':
        qc.x(0)
    if coloring[1] == '1':
        qc.x(1)
    
    # edge checking
    qc.cx(0,2)
    qc.cx(1,2)
    qc.measure(2,0)
    
    result = AerSimulator().run(qc, shots=1).result()
    counts = result.get_counts()
    
    output = list(counts.keys())[0][-1]  # last qubit
    
    print(qc)
    if output == '1':
        print("Edge connects different colors. Output:", output)
    else:
        print("Edge connects the same colors. Output:", output)



Vertex 0 is set to  0
Vertex 1 is set to  0
                  
q_0: ──■──────────
       │          
q_1: ──┼────■─────
     ┌─┴─┐┌─┴─┐┌─┐
q_2: ┤ X ├┤ X ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 
Edge connects the same colors. Output: 0

Vertex 0 is set to  0
Vertex 1 is set to  1
                       
q_0: ───────■──────────
     ┌───┐  │          
q_1: ┤ X ├──┼────■─────
     └───┘┌─┴─┐┌─┴─┐┌─┐
q_2: ─────┤ X ├┤ X ├┤M├
          └───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 
Edge connects different colors. Output: 1

Vertex 0 is set to  1
Vertex 1 is set to  0
     ┌───┐             
q_0: ┤ X ├──■──────────
     └───┘  │          
q_1: ───────┼────■─────
          ┌─┴─┐┌─┴─┐┌─┐
q_2: ─────┤ X ├┤ X ├┤M├
          └───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 
Edge connects different colors. Output: 1

Vertex 0 is set to  1
Vertex 1 is set to  1
     ┌───┐             
q_0: ┤ X ├──■──────────
     ├───┤  │          
q_1: ┤ X ├──┼────■──

## Designing an oracle for checking bipartiteness

In Grover's Algorithm, our aim is to find an element marked by the oracle. If we have an oracle that "marks" colorings in which the endpoints of all the edges are colored using a different color, then we can use Grover's search to find such colorings (if exists). Therefore, we need to define an oracle which will check and "mark" colorings which satisfy this property. 

As we described above, we use seperate qubits for each vertex and for each edge. The states of the qubits corresponding to vertices represent the colorings. The input to the oracle are the qubits representing the vertices, and the rest of the qubits will be used by the oracle. Before the oracle starts its computation, all qubits representing the edges and the output qubit are in $ \ket{0}$ state, and they should also be in $\ket{0} $ state at the end to be used again for the next oracle call.

We can summarize the computation by the oracle in four steps:

1. By using the XOR operator, we assign the appropriate Boolean value for each edge. 
2. Then, we use an additional output qubit, which is set to 1 if each edge has the value of 1, i.e., each edge connects two vertices with different colors. This part can be implemented by a multi-controlled $NOT$ operator (flip the value of target qubit only if all controlled qubits are in states $\ket{1}$). 

### Task 2

For the given graph below, implement the first two steps of the oracle described above. 

The first four qubits are used for vertices.

The next three qubits are used for edges.

The last qubit is used for the output.

Remark that the last qubit should be in state $ \ket{1} $ if the endpoints of all the edges are colored using a different color and $\ket{0} otherwise$.

Test your implementation with different colorings.

<img src="../images/bipartite.png" width="25%" align="center">

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import MCXGate

qc = QuantumCircuit(8, 1)

# All end points have different colors
qc.x(0)
qc.x(1)

# Not all end points have different colors
# qc.x(0)

# check 0-2 edge and store at 4th qubit


# check 0-3 edge and store at 5th qubit

# check 1-3 edge and store at 6th qubit


# check all edge qubits


job = AerSimulator().run(qc, shots=1)
counts = job.result().get_counts(qc)
print(counts)

if list(counts.keys())[0][0] == '1':
    print("All end points have different colors (graph is bipartite)")
else:
    print("Not all end points have different colors (graph is not bipartite)")

[click for our solution](02_max_cut_problem_solutions.ipynb#task2)

Let's continue with the remaining steps of the oracle.

3. The oracle flips the sign of the amplitude of the states which lead to $ \ket{1} $ as the output. This can be easily done by applying a $Z$ gate: when in state $\ket{1}$, the sign is flipped, and it does not change, otherwise. 
4. We reverse the whole computation done in the first 2 steps so that the oracle leaves all the qubits used in the computation in state $\ket{0}$. This can be done by reversing every quantum operator before the $Z$ gate. In this way, the only effect of the oracle is flipping the amplitude of the sign of the qubits representing the correct coloring. To implement this step, we will use the `inverse` function in Qiskit.

Now, our oracle is ready to be used as a part of Grover’s search algorithm. 

In the following code, we apply the oracle once and observe its effect. We start with an equal superposition of all possible coloring for the vertices (the first qubits).

In [ ]:
from qiskit import QuantumCircuit
from helpers import print_dirac

def oracle_computation(): #Note that we do not pass any parameters here
    temp_qc = QuantumCircuit(8)
    # check 0-2 edge and store at 4th qubit
    temp_qc.cx(0,4)
    temp_qc.cx(2,4)
    # check 0-3 edge and store at 5th qubit
    temp_qc.cx(0,5)
    temp_qc.cx(3,5)
    # check 1-3 edge and store at 6th qubit
    temp_qc.cx(1,6)
    temp_qc.cx(3,6)
    # check all edge qubits
    temp_qc.mcx([4,5,6], 7)
    return temp_qc #This is needed to be able to take its inverse later
    
def oracle(qc):

    qc.append(oracle_computation(), range(8)) #Apply oracle
    qc.z(7) #Phase flip
    qc.append(oracle_computation().inverse(),range(8)) #Uncompute oracle

qc = QuantumCircuit(8)
# Initialize to uniform superposition
for i in range(4):
    qc.h(i)
print("Initial state:")
print_dirac(qc)

# Apply oracle
oracle(qc)
print("State after applying oracle:")
print_dirac(qc)

Initial state:
0.25 |00000000⟩ + 0.25 |00000001⟩ + 0.25 |00000010⟩ + 0.25 |00000011⟩ + 0.25 |00000100⟩ + 0.25 |00000101⟩ + 0.25 |00000110⟩ + 0.25 |00000111⟩ + 0.25 |00001000⟩ + 0.25 |00001001⟩ + 0.25 |00001010⟩ + 0.25 |00001011⟩ + 0.25 |00001100⟩ + 0.25 |00001101⟩ + 0.25 |00001110⟩ + 0.25 |00001111⟩
State after oracle computation:
0.25 |00000000⟩ + 0.25 |00000001⟩ + 0.25 |00000010⟩ + -0.25 |00000011⟩ + 0.25 |00000100⟩ + 0.25 |00000101⟩ + 0.25 |00000110⟩ + 0.25 |00000111⟩ + 0.25 |00001000⟩ + 0.25 |00001001⟩ + 0.25 |00001010⟩ + 0.25 |00001011⟩ + -0.25 |00001100⟩ + 0.25 |00001101⟩ + 0.25 |00001110⟩ + 0.25 |00001111⟩


Note several important things here. The first four qubits are set to zero: these are edge checking and output qubit, and since they are set to zero, we can reuse them in further computation.

As observed from the outcome, only the sign of the following states are flipped:
$$
	|{0000}\rangle|{0011}\rangle  \text{ and } |{0000}\rangle|{1100}\rangle.
$$

In the first case, the vertices 0 and 1 are colored blue and the vertices 2 and 3 are colored red. In the second case, the first pair is colored red and the second one is colored blue.


### Task 3 

For the given graphs below, iterate Grover’s search algorithm 2 steps to find the correct colorings. (There are indeed $k=2$ possible colorings, and so the oracle should be applied $\frac{\pi}{4}\sqrt{\frac{2^4}{k}}\approx 2$ times.) 

You will use 9 qubits: 4 for vertices, 4 for edges, and 1 for the output qubit.

Observe which outcomes have the higher frequencies.


<img src="../images/completebipartite.png" width="25%" align="center">

In [ ]:
from qiskit import QuantumCircuit, transpile    
from qiskit.circuit.library import ZGate


def edge_check(a, b, c):
    yield CX(qq[a], qq[c])
    yield CX(qq[b], qq[c])

def oracle_computation():
    temp_qc = QuantumCircuit(9)
    # check 0-2 edge and store at 4th qubit
    
    # check 0-3 edge and store at 5th qubit
   
    # check 1-2 edge and store at 6th qubit
    
    # check 1-3 edge and store at 7th qubit
    

    # check all edge qubits

    return temp_qc
    
def oracle(qc):
    #Your code here

def grover_diffusion(qc):
    #Your code here

#Define quantum circuit
qc = QuantumCircuit(9, 4)

#Initialize to uniform superposition
for i in range(4):
    qc.h(i)

#Apply Grover iterations
for i in range(2):
    oracle(qc)
    grover_diffusion(qc)
    
qc.measure(range(4), range(4))
tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=1024)
counts = job.result().get_counts(qc)
print(counts)

[click for our solution](02_max_cut_problem_solutions.ipynb#task3)

<a id="task4"></a>
### Task 4

Repeat Task 3 for the following graph.

Is it possible to color the following graph so that all edges have endpoints with different colors? If not, what would you say in advance about the frequencies of the outcomes?


<img src="../images/nonbipartite.png" width="25%" align="center">

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate

def oracle():
    # Your code here

def oracle(qc):
    #Your code here
    
def grover_diffusion(qc):
    #Your code here

#Your code here

print(counts)

[click for our solution](02_max_cut_problem_solutions.ipynb#task4)

## Conclusion 


Here we implement Grover's search algorithm to find out whether a given graph can be colored so that endpoints of each edge have different colors and eventually to determine whether the given graph is bipartite or not. When the given graph is bipartite, then solving Max-Cut problem for the given graph becomes trivial. 

We should note that our algorithm takes $O(\sqrt{2^n})$ steps, but this problem can be solved classically in $O(n^2)$ steps. On the other hand, we see this application as a pedagogical example of how to design an oracle and then apply Grover’s search algorithm for a well-known problem on graphs. 

In the following notebooks, we will construct an oracle which solves Max-Cut problem for any arbitrary graph.